# 1D Dealiasing and the 2/3 Rule: Improved Walkthrough

This notebook demonstrates how pointwise multiplication in physical space creates non-physical Fourier modes (aliasing) and how Orszag's 2/3 rule prevents spurious mode fold-back in quadratic pseudo-spectral methods.

### What you will see
- Mathematical mechanics of alias folding in discrete Fourier space.
- Why high-frequency mode products fold back into low modes.
- How the 2/3 rule and zero-padding eliminate aliasing errors.
- An interactive mode explorer and an edge-case breakdown where output filtering alone fails.

## What is improved in this version

- Fixed syntax errors, especially missing multiplication operators.
- Centralized grid, masking, aliasing, and plotting helpers.
- Corrected the mask plotting call so the mask is not shifted twice.
- Added a noise floor to spectrum plots so round-off noise (about 1e-16) no longer looks like physical modes.
- Added diagnostics when a mode lies outside the 2/3 retained band and the dealiased product becomes zero.
- Made the zero-padding example explicit and noted the Nyquist-mode caveat.
- Added a fallback for environments without ipywidgets.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    from ipywidgets import interact, IntSlider
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.4
plt.rcParams['lines.markersize'] = 6

In [ ]:
def make_grid(N=64, L=2*np.pi):
    '''Return x grid, spacing, and FFT wavenumbers.'''
    x = np.linspace(0, L, N, endpoint=False)
    dx = L / N
    k = np.fft.fftfreq(N, d=dx) * 2*np.pi
    return x, dx, k

def nyquist(N):
    return N // 2

def two_thirds_mask(k, N):
    '''Return a mask that keeps |k| <= (2/3) * k_max.'''
    kmax = nyquist(N)
    cutoff = (2.0 / 3.0) * kmax
    mask = (np.abs(k) <= cutoff).astype(float)
    return mask, cutoff

def alias_wavenumber(q, N):
    '''Map integer wavenumbers to their aliases on an N-point grid.'''
    kmax = N // 2
    return ((np.asarray(q) + kmax) % N) - kmax

def stem_spectrum(ax, k, hat, N, title='', cutoff=None, color='C0', noise_floor=1e-12):
    '''Plot a centered amplitude spectrum, shade truncated zones, and hide round-off noise.'''
    k_shifted = np.fft.fftshift(k)
    amp = np.abs(np.fft.fftshift(hat)) / N
    peak = amp.max()
    amp = np.where(amp < noise_floor, 0.0, amp)

    ax.stem(k_shifted, amp, linefmt=color + '-', markerfmt=color + 'o', basefmt='k-')

    if cutoff is not None:
        max_k = np.max(k_shifted) + 1
        ax.axvspan(cutoff, max_k, color='red', alpha=0.12, label='Truncated zone')
        ax.axvspan(-max_k, -cutoff, color='red', alpha=0.12)
        ax.axvline(cutoff, color='r', linestyle='--', linewidth=1.2)
        ax.axvline(-cutoff, color='r', linestyle='--', linewidth=1.2)

    if peak < noise_floor:
        ax.text(0.5, 0.5, 'numerical noise only (max < 1e-12)', transform=ax.transAxes,
                ha='center', va='center', fontsize=11, color='gray')
        ax.set_ylim(0, 1)

    ax.set_title(title)
    ax.set_xlabel('Wavenumber k')
    ax.set_ylabel('|FFT| / N')

## 1. First experiment: a sum mode just above Nyquist

Let N=64, so k_max=32. Choose k1=20 and k2=15.

- Difference mode: k1 - k2 = 5, safely resolved.
- Sum mode: k1 + k2 = 35, exceeds k_max=32.

The sum mode folds to 35 - 64 = -29. Because the spatial field is real, the negative counterpart appears at +29 in the shifted spectrum.

In [ ]:
N = 64
L = 2*np.pi
x, dx, k = make_grid(N, L)
kmax = nyquist(N)
mask, cutoff = two_thirds_mask(k, N)

k1, k2 = 20, 15
u = np.cos(k1 * x)
a = np.cos(k2 * x)

sum_mode = k1 + k2
diff_mode = k1 - k2
sum_alias = int(alias_wavenumber(sum_mode, N))
neg_sum_alias = int(alias_wavenumber(-sum_mode, N))

print(f'N = {N}')
print(f'Nyquist k_max = {kmax}')
print(f'2/3 cutoff = {cutoff:.2f}')
print(f'Chosen modes: k1 = {k1}, k2 = {k2}')
print(f'Sum mode = {sum_mode}, difference mode = {diff_mode}')
print(f'Sum mode aliases to k = {sum_alias}')
print(f'Negative sum aliases to k = {neg_sum_alias}')

In [ ]:
p_aliased = a * u
p_aliased_hat = np.fft.fft(p_aliased)
exact_continuum_coarse = 0.5 * np.cos(sum_mode * x) + 0.5 * np.cos(diff_mode * x)

print('Max difference between pointwise product and continuous identity on grid:')
print(np.max(np.abs(p_aliased - exact_continuum_coarse)))
print('The grid samples match because the unresolved continuous mode is indistinguishable from its alias on this grid.')

## 2. Apply the 2/3 dealiasing mask

The 2/3 mask keeps modes where |k| <= (2/3) k_max. For N=64, modes up to |k| = 21 are retained.

Input filtering zeroes modes above the threshold before forming pointwise products. Output filtering strips high-frequency artifacts after multiplication. For a robust quadratic dealiasing strategy, use both.

In [ ]:
u_hat_clean = np.fft.fft(u) * mask
a_hat_clean = np.fft.fft(a) * mask

u_clean = np.fft.ifft(u_hat_clean).real
a_clean = np.fft.ifft(a_hat_clean).real

p_dealiased = a_clean * u_clean
p_dealiased_hat = np.fft.fft(p_dealiased) * mask
p_dealiased_space = np.fft.ifft(p_dealiased_hat).real

## 3. Visual comparison

Red-shaded regions represent modes beyond the 2/3 threshold. The aliased interpolant matches the grid samples but differs from the true continuum between grid points.

In [ ]:
x_fine = np.linspace(0, L, 1024, endpoint=False)
aliased_interpolant = 0.5 * np.cos(sum_alias * x_fine) + 0.5 * np.cos(diff_mode * x_fine)
exact_continuum_fine = 0.5 * np.cos(sum_mode * x_fine) + 0.5 * np.cos(diff_mode * x_fine)
resolved_exact_fine = 0.5 * np.cos(diff_mode * x_fine)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

stem_spectrum(axes[0, 0], k, p_aliased_hat, N, title='Aliased spectrum', cutoff=cutoff, color='C0')
axes[0, 0].legend()

stem_spectrum(axes[0, 1], k, p_dealiased_hat, N, title='2/3 dealiased spectrum', cutoff=cutoff, color='C2')
axes[0, 1].legend()

axes[1, 0].plot(x_fine, exact_continuum_fine, 'k--', label='True continuum')
axes[1, 0].plot(x_fine, aliased_interpolant, color='C0', alpha=0.75, label='Aliased interpolant')
axes[1, 0].plot(x, p_aliased, 'o', color='C3', markersize=4, label='Grid samples')
axes[1, 0].set_title('Spatial field: aliased vs true continuum')
axes[1, 0].set_xlabel('x')
axes[1, 0].set_ylabel('a(x) u(x)')
axes[1, 0].legend()

axes[1, 1].plot(x_fine, resolved_exact_fine, 'r--', label='Resolved exact mode (k=5)')
axes[1, 1].plot(x, p_dealiased_space, 's', color='green', markersize=4, label='Dealiased grid values')
axes[1, 1].set_title('Spatial field after full 2/3 dealiasing')
axes[1, 1].set_xlabel('x')
axes[1, 1].set_ylabel('Dealiased product')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 4. The 2/3 dealiasing mask

Modes in the shaded red zones are set to zero.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
stem_spectrum(ax, k, mask * N, N, title='2/3 dealiasing mask', cutoff=cutoff, color='C0')
ax.set_ylabel('Mask value')
ax.set_ylim(-0.1, 1.1)
plt.tight_layout()
plt.show()

## 5. Why exactly 2/3?

Let K = N/2 be the Nyquist limit. Suppose we keep modes up to M = (2/3) K.

When two retained modes interact, the largest product wavenumber is M + M = (4/3) K.

Because (4/3) K > K, it aliases on an N-point grid by subtracting 2K:

    k_alias = (4/3) K - 2K = -(2/3) K

The folded mode lands exactly at the outer edge of the retained band. Removing modes above (2/3) K prevents aliased quadratic products from contaminating the interior resolved band.

## 6. A dangerous case: aliasing directly into the low band

Consider k1 = k2 = 30 with N=64. These modes are representable but lie outside the 2/3 band.

Their sum is 60, which aliases to 60 - 64 = -4. Since |-4| is inside the retained low band, output masking alone fails: the spurious frequency is mistaken for a true low mode.

In [ ]:
N_demo = 64
x_demo, _, k_demo = make_grid(N_demo)
mask_demo, cutoff_demo = two_thirds_mask(k_demo, N_demo)

def product_spectrum_for_modes(k1_demo, k2_demo, mask_inputs=False, mask_output=False):
    u_demo = np.cos(k1_demo * x_demo)
    a_demo = np.cos(k2_demo * x_demo)

    if mask_inputs:
        u_demo = np.fft.ifft(np.fft.fft(u_demo) * mask_demo).real
        a_demo = np.fft.ifft(np.fft.fft(a_demo) * mask_demo).real

    p_demo = a_demo * u_demo
    p_demo_hat = np.fft.fft(p_demo)

    if mask_output:
        p_demo_hat = p_demo_hat * mask_demo

    return p_demo_hat

k1_test, k2_test = 30, 30
k_target = int(alias_wavenumber(k1_test + k2_test, N_demo))

hat_out = product_spectrum_for_modes(k1_test, k2_test, mask_inputs=False, mask_output=True)
k_demo_shifted = np.fft.fftshift(k_demo)
hat_out_shifted = np.fft.fftshift(hat_out)
spurious_amp = np.abs(hat_out_shifted[k_demo_shifted == k_target])[0] / N_demo

print(f'k1 = {k1_test}, k2 = {k2_test}')
print(f'Sum mode {k1_test + k2_test} aliases to k = {k_target}')
print(f'Spurious mode survives output masking alone: {spurious_amp > 1e-10}')

cases = [
    ('No dealiasing', False, False),
    ('Output mask only (fails)', False, True),
    ('Input mask only', True, False),
    ('Full 2/3 rule', True, True),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
for ax, (title, mi, mo) in zip(axes.flat, cases):
    hat = product_spectrum_for_modes(k1_test, k2_test, mask_inputs=mi, mask_output=mo)
    stem_spectrum(ax, k_demo, hat, N_demo, title=title, cutoff=cutoff_demo, color='C0')

axes[0, 0].legend()
plt.tight_layout()
plt.show()

## 7. Interactive mode explorer

Use the sliders to explore mode interactions. If ipywidgets is unavailable, a static example is shown instead.

### How to read the panels

- If a slider mode lies outside the retained band (|k| > 2/3 k_max), the input mask removes it and the dealiased product is zero. This is intentional: the 2/3 rule discards interactions involving truncated modes, so even a resolved difference mode disappears.
- Stems of size about 1e-16 are floating-point round-off, not physical modes. The plotting helper hides anything below 1e-12 and labels such panels.

In [ ]:
def compare_modes(k1_test, k2_test, N_test=64):
    x_test, _, k_test = make_grid(N_test)
    mask_test, cutoff_test = two_thirds_mask(k_test, N_test)

    u_test = np.cos(k1_test * x_test)
    a_test = np.cos(k2_test * x_test)

    p_raw_hat = np.fft.fft(a_test * u_test)

    u_clean_test = np.fft.ifft(np.fft.fft(u_test) * mask_test).real
    a_clean_test = np.fft.ifft(np.fft.fft(a_test) * mask_test).real
    p_clean_hat = np.fft.fft(a_clean_test * u_clean_test) * mask_test

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    stem_spectrum(axes[0], k_test, p_raw_hat, N_test, title=f'Raw spectrum: k1={k1_test}, k2={k2_test}', cutoff=cutoff_test, color='C0')
    stem_spectrum(axes[1], k_test, p_clean_hat, N_test, title='After full 2/3 dealiasing', cutoff=cutoff_test, color='C2')
    axes[0].legend()
    axes[1].legend()
    plt.tight_layout()
    plt.show()

    print(f'Sum mode {k1_test + k2_test} aliases to k = {int(alias_wavenumber(k1_test + k2_test, N_test))}')
    print(f'Difference mode {k1_test - k2_test} aliases to k = {int(alias_wavenumber(k1_test - k2_test, N_test))}')

    truncated = [kk for kk in (k1_test, k2_test) if kk > cutoff_test]
    if truncated:
        print(f'Note: mode(s) {truncated} lie outside the 2/3 retained band (|k| <= {cutoff_test:.2f}).')
        print('The input mask removes them, so the dealiased product is zero up to round-off.')
    if np.max(np.abs(p_clean_hat)) / N_test < 1e-12:
        print('Dealiased spectrum is pure floating-point noise (about 1e-16); the plot floor hides it.')

if HAS_WIDGETS:
    @interact(
        k1=IntSlider(min=0, max=31, step=1, value=18),
        k2=IntSlider(min=0, max=31, step=1, value=17)
    )
    def interactive_entry(k1, k2):
        compare_modes(k1, k2)
else:
    print('ipywidgets is not available; running a static example instead.')
    compare_modes(18, 17)

## 8. Alternative: the 3/2 zero-padding rule

Instead of truncating modes before multiplication, pad Fourier vectors with zeros up to M = 3N/2. Pointwise multiplication is then evaluated on the fine M-point grid, preventing high-frequency convolution from wrapping back into the original modes.

The helper below keeps non-Nyquist modes only. This is enough for the examples and avoids the subtle even-N Nyquist ambiguity. In production code, use a battle-tested padding routine or split the Nyquist mode carefully.

In [ ]:
def pad_spectrum_dealiasing(hat, N_pad):
    '''Pad an N-point FFT to N_pad points for dealiased multiplication.'''
    N = len(hat)
    out = np.zeros(N_pad, dtype=complex)
    n_pos = N // 2

    # Keep non-Nyquist modes. This avoids the even-N Nyquist ambiguity.
    out[:n_pos] = hat[:n_pos]
    if n_pos > 1:
        out[-(n_pos - 1):] = hat[n_pos + 1:]

    return out * (N_pad / N)

def truncate_padded_spectrum(hat_pad, N):
    '''Truncate a padded FFT back to N modes and restore the N-point FFT scaling.'''
    N_pad = len(hat_pad)
    out = np.zeros(N, dtype=complex)
    n_pos = N // 2

    out[:n_pos] = hat_pad[:n_pos]
    if n_pos > 1:
        out[n_pos + 1:] = hat_pad[-(n_pos - 1):]

    # The Nyquist bin, out[n_pos], is intentionally left at zero.
    return out * (N / N_pad)

N = 64
N_pad = int(1.5 * N)
x, _, k = make_grid(N)

u = np.cos(20 * x)
a = np.cos(15 * x)

u_hat = np.fft.fft(u)
a_hat = np.fft.fft(a)

u_hat_pad = pad_spectrum_dealiasing(u_hat, N_pad)
a_hat_pad = pad_spectrum_dealiasing(a_hat, N_pad)

u_pad = np.fft.ifft(u_hat_pad).real
a_pad = np.fft.ifft(a_hat_pad).real
p_pad = u_pad * a_pad

p_pad_hat = np.fft.fft(p_pad)
mask, cutoff = two_thirds_mask(k, N)
p_clean_pad_hat = truncate_padded_spectrum(p_pad_hat, N) * mask

p_ref_23_hat = np.fft.fft(
    np.fft.ifft(np.fft.fft(a) * mask).real *
    np.fft.ifft(np.fft.fft(u) * mask).real
) * mask

print(f'Zero-padded product computed on N_pad = {N_pad}')
print('Max coefficient difference between zero-padding plus mask and full 2/3 rule:')
print(np.max(np.abs(p_clean_pad_hat - p_ref_23_hat)))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
stem_spectrum(axes[0], k, np.fft.fft(a * u), N, title='Aliased product on original grid', cutoff=cutoff, color='C0')
stem_spectrum(axes[1], k, p_clean_pad_hat, N, title='3/2 zero-padding plus retained mask', cutoff=cutoff, color='C2')
axes[0].legend()
axes[1].legend()
plt.tight_layout()
plt.show()

## 9. Exercises

- Change grid size N and verify the alias formula across arbitrary modes.
- Extend the derivation to cubic nonlinearities, u^3, and determine the required cutoff fraction. Hint: consider 1/2 rule.
- Benchmark the runtime efficiency of the 2/3 mask versus 3/2 zero-padding for large N.

## Summary

Aliasing is an inherent artifact of evaluating nonlinear products on discrete spatial grids. Applying Orszag's 2/3 rule or 3/2 zero-padding guarantees that high-frequency interactions do not corrupt resolved physical modes in pseudo-spectral simulations.